# 九州8空港 Traffic Patterns

空港を選び、4場周の運用profile、平面経路・高度profileを確認し、KML / KMZを出力します。
既存のReferencePath builderを使う幾何モデルです。風・飛行力学・地形クリアランスの再現ではありません。
RJFM固有のShort Downwindは `miyazaki_traffic_patterns.ipynb` を使用してください。

## 1. Setup

In [ ]:
import json
import math
import os
from dataclasses import asdict
from html import escape
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import HTML, Markdown, display
from sr22_course_simulator.examples.kyushu_traffic_patterns import (
    KYUSHU_AIRPORTS,
    build_kyushu_traffic_patterns,
    write_kyushu_traffic_pattern_exports,
)
from sr22_course_simulator.examples.rjf_traffic_patterns import pattern_identity
from sr22_course_simulator.geometry import enu_displacement
from sr22_course_simulator.units import metres_to_feet

repository_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
artifact_dir = Path(os.environ.get("SR22_ARTIFACT_DIR", repository_root / "artifacts")) / "kyushu-traffic-patterns"
print("Airport choices:", ", ".join(KYUSHU_AIRPORTS))

## 2. Airport selection / independent switches

`AIRPORT` を RJFM / RJFS / RJFO / RJFK / RJFT / RJFG / RJFU / RJFC から選び、以降を順に再実行してください。
5つのBooleanは独立しています。通常場周は常に表示・出力し、選択したCircle / 270を別componentとして加えます。
`EXPORT_ALL_AIRPORTS=True` は同じswitch設定で8空港の個別KMZと九州全体KMZも生成します。

In [ ]:
AIRPORT = "RJFM"
MAKE_CIRCLE_BEFORE_DOWNWIND = True
MAKE_270_BEFORE_DOWNWIND = True
MAKE_CIRCLE_MIDDLE_DOWNWIND = True
MAKE_CIRCLE_BEFORE_BASE = False
MAKE_270_BEFORE_BASE = True
EXPORT_ALL_AIRPORTS = True

## 3. 4 Traffic Patterns / source metadata

高度はft MSLです。preferredはmetadataであり、反対側の場周も保持します。
高度・降下開始点は既存の空港profileから読み込みます。
7空港の運用profileは **2026-09-03 AIRAC AMDTを情報基準時点とするsource snapshot** です。
これは各資料の改正・発効日とは別で、以降の改正は自動反映しません。
詳しくは [source snapshotの定義](../docs/data-sources.md#information-baseline) を参照してください。

In [ ]:
switches = {
    "make_circle_before_downwind": MAKE_CIRCLE_BEFORE_DOWNWIND,
    "make_270_before_downwind": MAKE_270_BEFORE_DOWNWIND,
    "make_circle_middle_downwind": MAKE_CIRCLE_MIDDLE_DOWNWIND,
    "make_circle_before_base": MAKE_CIRCLE_BEFORE_BASE,
    "make_270_before_base": MAKE_270_BEFORE_BASE,
}
pairs = build_kyushu_traffic_patterns(AIRPORT, **switches)
profile_rows = [
    (pattern_identity(spec), spec.altitude_ft, spec.descent_start.value, spec.preferred)
    for spec, components in pairs
]
display(Markdown(
    "| Pattern | Altitude [ft MSL] | Descent start | Preferred |\n"
    "| --- | ---: | --- | --- |\n" +
    "\n".join(f"| {identity} | {altitude:g} | {descent} | {preferred} |"
              for identity, altitude, descent, preferred in profile_rows)
))
for spec, components in pairs:
    provenance = {
        "notes": spec.notes,
        "operational_source": asdict(spec.source),
        "airport_source": asdict(spec.airport.source),
        "runway_source": asdict(spec.runway.source),
    }
    display(HTML(
        f"<details><summary>{escape(pattern_identity(spec))}: provenance / notes</summary>"
        f"<pre>{escape(json.dumps(provenance, ensure_ascii=False, indent=2))}</pre></details>"
    ))

## 4. 平面経路 / 高度profile

左列はRWY Center Point基準のEast / North [NM]、右列は各componentの始点からの水平沿程距離 [NM] と高度 [ft MSL] です。
独立componentの距離原点はそれぞれ0であり、通常場周の距離軸へ連結していません。色は左右の図で対応します。

In [ ]:
figure, axes = plt.subplots(4, 2, figsize=(15, 19), constrained_layout=True)
for row, (spec, components) in enumerate(pairs):
    ground, altitude = axes[row]
    origin = spec.runway.center_point
    for index, path in enumerate(components):
        points = path.points()
        xy = [enu_displacement(origin, p.position) for p in points]
        distance_nm = [0.0]
        for a, b in zip(points, points[1:]):
            distance_nm.append(distance_nm[-1] + math.hypot(*enu_displacement(a.position, b.position)) / 1852)
        label = path.name.removeprefix(pattern_identity(spec) + " ")
        style = {"color": f"C{index}", "linewidth": 1.8 if index == 0 else 1.1, "label": label}
        ground.plot([x / 1852 for x, y in xy], [y / 1852 for x, y in xy], **style)
        altitude.plot(distance_nm, [metres_to_feet(p.altitude_m) for p in points], **style)
    runway_xy = [enu_displacement(origin, point) for point in (spec.runway.threshold_a, spec.runway.threshold_b)]
    ground.plot([x / 1852 for x, y in runway_xy], [y / 1852 for x, y in runway_xy], color="black", linewidth=3, label="Runway")
    ground.set_aspect("equal", adjustable="datalim")
    ground.set(xlabel="East from RWY center [NM]", ylabel="North from RWY center [NM]", title=pattern_identity(spec))
    altitude.set(xlabel="Distance from component start [NM]", ylabel="Altitude MSL [ft]", title=f"{spec.altitude_ft:g} ft / {spec.descent_start.value}")
    ground.legend(fontsize=8, loc="upper left")
    ground.grid(True, alpha=0.3)
    altitude.grid(True, alpha=0.3)
plt.show()

## 5. KML / KMZ export

選択空港のraw KML（4場周＋結合）と `<ICAO>_TRAFFIC_PATTERNS.kmz` を保存します。
全空港出力を有効にすると、8空港のraw KML・個別KMZに加え `KYUSHU_TRAFFIC_PATTERNS.kmz` を保存します。
Google Earthでは空港 → RWY → LEFT/RIGHT → componentを展開できます。
KML座標はlongitude / latitude / absolute MSL altitude [m]。各Placemarkに運用・空港・滑走路の出典を保持します。

In [ ]:
written = write_kyushu_traffic_pattern_exports(
    artifact_dir, None if EXPORT_ALL_AIRPORTS else AIRPORT, **switches
)
print(f"Saved {len(written)} files to {artifact_dir}")
for path in written:
    if path.suffix == ".kmz" or path.name.startswith(AIRPORT.strip().upper() + "_"):
        print(path.name)

## 6. Checks

4場周と選択component数、生成ファイルを確認します。
KMZは既存KMLのpackagingであり、別の経路計算は行いません。

In [ ]:
assert len(pairs) == 4
assert all(len(components) == 1 + sum(switches.values()) for spec, components in pairs)
assert all(path.is_file() and path.stat().st_size > 0 for path in written)
expected_airports = KYUSHU_AIRPORTS if EXPORT_ALL_AIRPORTS else (pairs[0][0].airport.icao,)
assert {path.name for path in written if path.suffix == ".kmz"} == (
    {f"{icao}_TRAFFIC_PATTERNS.kmz" for icao in expected_airports}
    | ({"KYUSHU_TRAFFIC_PATTERNS.kmz"} if EXPORT_ALL_AIRPORTS else set())
)
print("4 patterns, independent components, and export files: OK")